In [1]:
# ── MASTER IMPORT CELL — Phase 4: Model Training & Evaluation ─────────────────
# All libraries needed across Steps 4.1 through 4.5

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve

import lightgbm as lgb

from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")



In [2]:

# Loading the process data from feature ready file from Phase - 3. 

df = pd.read_csv('../data/processed/application_train_processed.csv')

print(f"Processed data loaded : {df.shape}")
print(f"Expected (307511 , 105)")

Processed data loaded : (307511, 105)
Expected (307511 , 105)


In [3]:
# ── LOAD FEATURE LIST AND SEPARATE X, y ───────────────────────────────────────
# Read the exact 102 features confirmed in Step 3.5

with open('../docs/feature_list.txt', 'r') as f:
    lines = f.readlines()

# skip header lines and extract only feature list 
final_features = [line.strip() for line in lines
                  if line.strip() and not line.startswith('Final')
                  and not line.startswith('Generated')]

print(f"Features loaded from docs/feature_list.txt : {len(final_features)}")
print(f"Expected (102)")

# Seperate features (X) and target (y)
X = df[final_features]
y = df['TARGET']    

ids = df['SK_ID_CURR']
print(f"X shape : {X.shape}")
print(f"Y shape : {y.shape}")
print(f"Overall default rate :{y.mean()*100:.2f}%")

Features loaded from docs/feature_list.txt : 102
Expected (102)
X shape : (307511, 102)
Y shape : (307511,)
Overall default rate :8.07%


In [4]:
print(lines)

['Final Feature List ï¿½ 102 features\n', 'Generated: Step 3.5 Feature Selection\n', '\n', 'NAME_CONTRACT_TYPE\n', 'FLAG_OWN_CAR\n', 'FLAG_OWN_REALTY\n', 'CNT_CHILDREN\n', 'AMT_INCOME_TOTAL\n', 'AMT_CREDIT\n', 'AMT_ANNUITY\n', 'AMT_GOODS_PRICE\n', 'NAME_EDUCATION_TYPE\n', 'REGION_POPULATION_RELATIVE\n', 'DAYS_BIRTH\n', 'DAYS_EMPLOYED\n', 'DAYS_REGISTRATION\n', 'DAYS_ID_PUBLISH\n', 'OWN_CAR_AGE\n', 'FLAG_WORK_PHONE\n', 'OCCUPATION_TYPE\n', 'CNT_FAM_MEMBERS\n', 'REGION_RATING_CLIENT\n', 'REGION_RATING_CLIENT_W_CITY\n', 'HOUR_APPR_PROCESS_START\n', 'REG_REGION_NOT_WORK_REGION\n', 'LIVE_REGION_NOT_WORK_REGION\n', 'REG_CITY_NOT_LIVE_CITY\n', 'REG_CITY_NOT_WORK_CITY\n', 'LIVE_CITY_NOT_WORK_CITY\n', 'ORGANIZATION_TYPE\n', 'EXT_SOURCE_1\n', 'EXT_SOURCE_2\n', 'EXT_SOURCE_3\n', 'APARTMENTS_MEDI\n', 'BASEMENTAREA_MEDI\n', 'YEARS_BEGINEXPLUATATION_MEDI\n', 'YEARS_BUILD_MEDI\n', 'COMMONAREA_MEDI\n', 'ELEVATORS_MEDI\n', 'ENTRANCES_MEDI\n', 'FLOORSMAX_MEDI\n', 'FLOORSMIN_MEDI\n', 'LANDAREA_MEDI\n', '

In [5]:
# ── STRATIFIED TRAIN / VALIDATION / TEST SPLIT ────────────────────────────────
# 60% train, 20% validation, 20% test
# Stratify on TARGET to preserve 8.07% default rate in every split

# Step 1 — Split off the test set first (20%)
X_temp, X_test, y_temp, y_test, ids_temp, ids_test = train_test_split(
    X, y, ids,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Step 2 — Split the remaining 80% into train (60% of total) and validation (20% of total)
# 0.25 of the remaining 80% = 20% of the original total
X_train, X_val, y_train, y_val, ids_train, ids_val = train_test_split(
    X_temp, y_temp, ids_temp,
    test_size=0.25,
    stratify=y_temp,
    random_state=42
)

print("=== SPLIT COMPLETE ===\n")
print(f"Train:      {X_train.shape[0]:>7,} rows  ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation: {X_val.shape[0]:>7,} rows  ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test:       {X_test.shape[0]:>7,} rows  ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Total:      {X_train.shape[0]+X_val.shape[0]+X_test.shape[0]:>7,} rows")

print(f"\n=== DEFAULT RATE VERIFICATION (should all be ~8.07%) ===")
print(f"Overall:    {y.mean()*100:.2f}%")
print(f"Train:      {y_train.mean()*100:.2f}%")
print(f"Validation: {y_val.mean()*100:.2f}%")
print(f"Test:       {y_test.mean()*100:.2f}%")

=== SPLIT COMPLETE ===

Train:      184,506 rows  (60.0%)
Validation:  61,502 rows  (20.0%)
Test:        61,503 rows  (20.0%)
Total:      307,511 rows

=== DEFAULT RATE VERIFICATION (should all be ~8.07%) ===
Overall:    8.07%
Train:      8.07%
Validation: 8.07%
Test:       8.07%


In [6]:
# ── SAVE SPLITS TO DISK ────────────────────────────────────────────────────────
# Each split saved separately — the standard handoff pattern for Phase 4

import os
os.makedirs('../data/processed/splits', exist_ok=True)

# Combine X, y, and ids for each split before saving
train_df = X_train.copy()
train_df['TARGET'] = y_train
train_df['SK_ID_CURR'] = ids_train

val_df = X_val.copy()
val_df['TARGET'] = y_val
val_df['SK_ID_CURR'] = ids_val

test_df = X_test.copy()
test_df['TARGET'] = y_test
test_df['SK_ID_CURR'] = ids_test

train_df.to_csv('../data/processed/splits/train.csv', index=False)
val_df.to_csv('../data/processed/splits/validation.csv', index=False)
test_df.to_csv('../data/processed/splits/test.csv', index=False)

print("=== SPLITS SAVED ===\n")
print(f"train.csv:      {train_df.shape}")
print(f"validation.csv: {val_df.shape}")
print(f"test.csv:       {test_df.shape}")

print(f"\nSaved to: data/processed/splits/")
print(f"All three files gitignored (data/processed/ already excluded)")

=== SPLITS SAVED ===

train.csv:      (184506, 104)
validation.csv: (61502, 104)
test.csv:       (61503, 104)

Saved to: data/processed/splits/
All three files gitignored (data/processed/ already excluded)


In [7]:
# ── IMPUTE MISSING VALUES ─────────────────────────────────────────────────────
# Logistic regression requires complete numerical input — no NaN allowed
# Median imputation chosen for robustness against outliers (Step 2.4 findings)
# IS_MISSING flags already preserve the "was originally missing" signal

imputer = SimpleImputer(strategy='median')

# Fit the imputer on TRAINING data only — never fit on validation or test
# This prevents validation/test information from leaking into imputation values
X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train), 
    columns=X_train.columns, 
    index=X_train.index
)

# Apply the SAME fitted imputer to validation — using train medians, not val medians
X_val_imputed = pd.DataFrame(
    imputer.transform(X_val), 
    columns=X_val.columns, 
    index=X_val.index
)

print("=== IMPUTATION COMPLETE ===\n")
print(f"Missing values in X_train_imputed: {X_train_imputed.isnull().sum().sum()}")
print(f"Missing values in X_val_imputed:   {X_val_imputed.isnull().sum().sum()}")
print(f"\nX_train_imputed shape: {X_train_imputed.shape}")
print(f"X_val_imputed shape:   {X_val_imputed.shape}")

=== IMPUTATION COMPLETE ===

Missing values in X_train_imputed: 0
Missing values in X_val_imputed:   0

X_train_imputed shape: (184506, 102)
X_val_imputed shape:   (61502, 102)


In [8]:
# ── FEATURE SCALING ────────────────────────────────────────────────────────────
# Logistic regression is sensitive to feature scale unlike LightGBM
# Standardise so every feature has mean=0, std=1 before training

scaler = StandardScaler()

# Fit on training data only — same discipline as imputation
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_imputed),
    columns=X_train_imputed.columns,
    index=X_train_imputed.index
)

X_val_scaled = pd.DataFrame(
    scaler.transform(X_val_imputed),
    columns=X_val_imputed.columns,
    index=X_val_imputed.index
)

print("=== SCALING COMPLETE ===\n")
print(f"AMT_CREDIT before scaling — train min/max: "
      f"{X_train_imputed['AMT_CREDIT'].min():.0f} / "
      f"{X_train_imputed['AMT_CREDIT'].max():.0f}")
print(f"AMT_CREDIT after scaling  — train min/max: "
      f"{X_train_scaled['AMT_CREDIT'].min():.2f} / "
      f"{X_train_scaled['AMT_CREDIT'].max():.2f}")

# ── TRAIN LOGISTIC REGRESSION ─────────────────────────────────────────────────
# class_weight='balanced' handles the 8.07% class imbalance
# without it, the model would be biased toward predicting the majority class

logreg = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

logreg.fit(X_train_scaled, y_train)

print(f"\n=== MODEL TRAINED ===")
print(f"Converged: {logreg.n_iter_[0] < 1000}")
print(f"Iterations used: {logreg.n_iter_[0]}")

=== SCALING COMPLETE ===

AMT_CREDIT before scaling — train min/max: 45000 / 4050000
AMT_CREDIT after scaling  — train min/max: -1.38 / 8.56

=== MODEL TRAINED ===
Converged: True
Iterations used: 52


In [9]:
# ── GENERATE PREDICTIONS ──────────────────────────────────────────────────────
# predict_proba returns probability for both classes — we want class 1 (default)

train_probs = logreg.predict_proba(X_train_scaled)[:, 1]
val_probs   = logreg.predict_proba(X_val_scaled)[:, 1]

print("=== PREDICTED PROBABILITY DISTRIBUTION ===\n")
print(f"Validation set predicted probabilities:")
print(f"  Min:    {val_probs.min():.4f}")
print(f"  Max:    {val_probs.max():.4f}")
print(f"  Mean:   {val_probs.mean():.4f}")
print(f"  Median: {np.median(val_probs):.4f}")

# ── CALCULATE AUC-ROC ─────────────────────────────────────────────────────────
train_auc = roc_auc_score(y_train, train_probs)
val_auc   = roc_auc_score(y_val, val_probs)

print(f"\n=== AUC-ROC ===")
print(f"Train AUC-ROC:      {train_auc:.4f}")
print(f"Validation AUC-ROC: {val_auc:.4f}")

# ── CALCULATE GINI COEFFICIENT ────────────────────────────────────────────────
# Gini = 2 * AUC - 1
train_gini = 2 * train_auc - 1
val_gini   = 2 * val_auc - 1

print(f"\n=== GINI COEFFICIENT ===")
print(f"Train Gini:      {train_gini:.4f}")
print(f"Validation Gini: {val_gini:.4f}")

# ── CALCULATE KS STATISTIC ────────────────────────────────────────────────────
# Maximum separation between cumulative distributions of the two classes
fpr, tpr, thresholds = roc_curve(y_val, val_probs)
ks_stat = max(tpr - fpr)
ks_threshold = thresholds[np.argmax(tpr - fpr)]

print(f"\n=== KS STATISTIC ===")
print(f"Validation KS: {ks_stat:.4f}")
print(f"KS occurs at threshold: {ks_threshold:.4f}")

# ── CALCULATE AUC-PR ──────────────────────────────────────────────────────────
from sklearn.metrics import average_precision_score
val_auc_pr = average_precision_score(y_val, val_probs)

print(f"\n=== AUC-PR (Average Precision) ===")
print(f"Validation AUC-PR: {val_auc_pr:.4f}")
print(f"(Baseline/random AUC-PR for this imbalance would be ~0.0807)")

=== PREDICTED PROBABILITY DISTRIBUTION ===

Validation set predicted probabilities:
  Min:    0.0242
  Max:    1.0000
  Mean:   0.4233
  Median: 0.3998

=== AUC-ROC ===
Train AUC-ROC:      0.7483
Validation AUC-ROC: 0.7453

=== GINI COEFFICIENT ===
Train Gini:      0.4967
Validation Gini: 0.4906

=== KS STATISTIC ===
Validation KS: 0.3679
KS occurs at threshold: 0.5020

=== AUC-PR (Average Precision) ===
Validation AUC-PR: 0.2221
(Baseline/random AUC-PR for this imbalance would be ~0.0807)


In [10]:
# ── COEFFICIENT INTERPRETATION TABLE ──────────────────────────────────────────
coef_df = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'Coefficient': logreg.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print("=== TOP 15 COEFFICIENTS BY ABSOLUTE MAGNITUDE ===\n")
print(coef_df.head(15).to_string(index=False))

coef_df.to_csv('../docs/logreg_coefficients.csv', index=False)
print(f"\nSaved to docs/logreg_coefficients.csv")

# Save baseline metrics for comparison in Step 4.5
baseline_metrics = {
    'model': 'Logistic Regression',
    'train_auc': train_auc,
    'val_auc': val_auc,
    'train_gini': train_gini,
    'val_gini': val_gini,
    'val_ks': ks_stat,
    'val_auc_pr': val_auc_pr
}
pd.DataFrame([baseline_metrics]).to_csv('../docs/baseline_metrics.csv', index=False)
print(f"Saved to docs/baseline_metrics.csv")

=== TOP 15 COEFFICIENTS BY ABSOLUTE MAGNITUDE ===

                    Feature  Coefficient
            AMT_GOODS_PRICE    -0.441670
            EXT_SOURCE_MEAN    -0.297639
                 AMT_CREDIT     0.293352
   OBS_30_CNT_SOCIAL_CIRCLE     0.203785
          ANNUITY_TO_INCOME     0.196297
               EXT_SOURCE_3    -0.196187
   OBS_60_CNT_SOCIAL_CIRCLE    -0.196182
        EXT_SOURCE_WEIGHTED    -0.184845
          ANNUITY_TO_CREDIT    -0.155342
          ORGANIZATION_TYPE     0.150689
REGION_RATING_CLIENT_W_CITY     0.140558
             DEBT_TO_INCOME    -0.140553
            FLAG_DOCUMENT_3     0.135949
                AMT_ANNUITY     0.115273
        NAME_EDUCATION_TYPE    -0.115265

Saved to docs/logreg_coefficients.csv
Saved to docs/baseline_metrics.csv


In [11]:
# ── PREPARE DATA FOR LIGHTGBM ─────────────────────────────────────────────────
# No imputation, no scaling needed — LightGBM handles NaN and raw scale natively
# Reuse X_train, y_train, X_val, y_val from Step 4.1 — the UNIMPUTED versions

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"Missing values in X_train (expected, LightGBM handles natively): "
      f"{X_train.isnull().sum().sum()}")

# Calculate scale_pos_weight — the LightGBM equivalent of class_weight='balanced'
# Ratio of negative class count to positive class count
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nscale_pos_weight calculated: {scale_pos_weight:.4f}")
print(f"(Ratio of repayers to defaulters in training set)")

# LightGBM Dataset objects — more memory-efficient than raw dataframes
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val   = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

print(f"\nLightGBM Dataset objects created")

X_train shape: (184506, 102)
X_val shape:   (61502, 102)
Missing values in X_train (expected, LightGBM handles natively): 2134616

scale_pos_weight calculated: 11.3871
(Ratio of repayers to defaulters in training set)

LightGBM Dataset objects created


In [12]:
# ── LIGHTGBM HYPERPARAMETERS ──────────────────────────────────────────────────
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,              # complexity control — max leaves per tree
    'learning_rate': 0.05,         # step size — smaller = more careful learning
    'feature_fraction': 0.9,       # use 90% of features per tree — reduces overfitting
    'bagging_fraction': 0.8,       # use 80% of rows per tree — reduces overfitting
    'bagging_freq': 5,             # perform bagging every 5 iterations
    'scale_pos_weight': scale_pos_weight,
    'verbosity': -1,
    'seed': 42
}

print("=== TRAINING LIGHTGBM WITH EARLY STOPPING ===\n")

evals_result = {}

lgbm_model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,          # generous ceiling — early stopping will cut this short
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.record_evaluation(evals_result)
    ]
)

print(f"\n=== TRAINING COMPLETE ===")
print(f"Best iteration: {lgbm_model.best_iteration}")
print(f"Best validation AUC: {lgbm_model.best_score['valid']['auc']:.4f}")

=== TRAINING LIGHTGBM WITH EARLY STOPPING ===

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[240]	train's auc: 0.826454	valid's auc: 0.761733

=== TRAINING COMPLETE ===
Best iteration: 240
Best validation AUC: 0.7617


In [13]:
# ── FULL METRIC EVALUATION FOR LIGHTGBM ───────────────────────────────────────

# Generate predictions using the best iteration (early stopping already applied this)
lgbm_train_probs = lgbm_model.predict(X_train, num_iteration=lgbm_model.best_iteration)
lgbm_val_probs   = lgbm_model.predict(X_val, num_iteration=lgbm_model.best_iteration)

# AUC-ROC (already have this from training, recalculating for consistency)
lgbm_train_auc = roc_auc_score(y_train, lgbm_train_probs)
lgbm_val_auc   = roc_auc_score(y_val, lgbm_val_probs)

# Gini
lgbm_train_gini = 2 * lgbm_train_auc - 1
lgbm_val_gini   = 2 * lgbm_val_auc - 1

# KS Statistic
fpr, tpr, thresholds = roc_curve(y_val, lgbm_val_probs)
lgbm_ks_stat = max(tpr - fpr)
lgbm_ks_threshold = thresholds[np.argmax(tpr - fpr)]

# AUC-PR
lgbm_val_auc_pr = average_precision_score(y_val, lgbm_val_probs)

print("=== LIGHTGBM FULL METRICS ===\n")
print(f"Train AUC-ROC:      {lgbm_train_auc:.4f}")
print(f"Validation AUC-ROC: {lgbm_val_auc:.4f}")
print(f"\nTrain Gini:      {lgbm_train_gini:.4f}")
print(f"Validation Gini: {lgbm_val_gini:.4f}")
print(f"\nValidation KS: {lgbm_ks_stat:.4f}")
print(f"KS threshold:  {lgbm_ks_threshold:.4f}")
print(f"\nValidation AUC-PR: {lgbm_val_auc_pr:.4f}")

print(f"\n=== COMPARISON: LOGISTIC REGRESSION vs LIGHTGBM (Validation) ===")
print(f"{'Metric':<20}{'LogReg':>12}{'LightGBM':>12}{'Improvement':>14}")
print(f"{'Gini':<20}{val_gini:>12.4f}{lgbm_val_gini:>12.4f}{lgbm_val_gini-val_gini:>+14.4f}")
print(f"{'KS':<20}{ks_stat:>12.4f}{lgbm_ks_stat:>12.4f}{lgbm_ks_stat-ks_stat:>+14.4f}")
print(f"{'AUC-PR':<20}{val_auc_pr:>12.4f}{lgbm_val_auc_pr:>12.4f}{lgbm_val_auc_pr-val_auc_pr:>+14.4f}")

=== LIGHTGBM FULL METRICS ===

Train AUC-ROC:      0.8265
Validation AUC-ROC: 0.7617

Train Gini:      0.6529
Validation Gini: 0.5235

Validation KS: 0.3883
KS threshold:  0.5139

Validation AUC-PR: 0.2398

=== COMPARISON: LOGISTIC REGRESSION vs LIGHTGBM (Validation) ===
Metric                    LogReg    LightGBM   Improvement
Gini                      0.4906      0.5235       +0.0329
KS                        0.3679      0.3883       +0.0204
AUC-PR                    0.2221      0.2398       +0.0177


In [14]:
# ── DEFINITIVE FEATURE IMPORTANCE ────────────────────────────────────────────
importance_final = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': lgbm_model.feature_importance(importance_type='gain')
}).sort_values('Importance', ascending=False)

print("=== TOP 20 FEATURES — FINAL LIGHTGBM MODEL ===\n")
print(importance_final.head(20).to_string(index=False))

# Compare against Step 3.5 preliminary ranking
print(f"\n=== ZERO IMPORTANCE FEATURES IN FINAL MODEL ===")
zero_imp_final = importance_final[importance_final['Importance'] == 0]
print(f"Count: {len(zero_imp_final)}")

=== TOP 20 FEATURES — FINAL LIGHTGBM MODEL ===

               Feature    Importance
       EXT_SOURCE_MEAN 206270.910503
   EXT_SOURCE_WEIGHTED 187560.594244
     ANNUITY_TO_CREDIT  73405.591751
       CREDIT_TO_GOODS  30708.188898
          EXT_SOURCE_1  29610.677362
       OCCUPATION_TYPE  24062.253971
         DAYS_EMPLOYED  23606.361374
          EXT_SOURCE_3  22371.225658
           AMT_ANNUITY  21786.096636
     ORGANIZATION_TYPE  17841.342442
            DAYS_BIRTH  17332.654404
       AMT_GOODS_PRICE  16695.833824
          EXT_SOURCE_2  16277.819860
      EMPLOYMENT_YEARS  15525.033106
           OWN_CAR_AGE  15460.807533
   NAME_EDUCATION_TYPE  13880.068714
            AMT_CREDIT  13112.469769
       DAYS_ID_PUBLISH  12540.581686
     ANNUITY_TO_INCOME  12404.409376
DAYS_LAST_PHONE_CHANGE  12315.768377

=== ZERO IMPORTANCE FEATURES IN FINAL MODEL ===
Count: 4


In [15]:
# ── SAVE THE TRAINED MODEL ─────────────────────────────────────────────────────
import os
os.makedirs('../models', exist_ok=True)

lgbm_model.save_model('../models/lightgbm_credit_model.txt')
print("Model saved to models/lightgbm_credit_model.txt")

# Save final feature importance
importance_final.to_csv('../docs/feature_importance_final.csv', index=False)
print("Feature importance saved to docs/feature_importance_final.csv")

# Save LightGBM metrics for Step 4.5 comparison table
lgbm_metrics = {
    'model': 'LightGBM',
    'train_auc': lgbm_train_auc,
    'val_auc': lgbm_val_auc,
    'train_gini': lgbm_train_gini,
    'val_gini': lgbm_val_gini,
    'val_ks': lgbm_ks_stat,
    'val_auc_pr': lgbm_val_auc_pr,
    'best_iteration': lgbm_model.best_iteration
}
pd.DataFrame([lgbm_metrics]).to_csv('../docs/lightgbm_metrics.csv', index=False)
print("LightGBM metrics saved to docs/lightgbm_metrics.csv")

Model saved to models/lightgbm_credit_model.txt
Feature importance saved to docs/feature_importance_final.csv
LightGBM metrics saved to docs/lightgbm_metrics.csv
